# 18.4 Data labeling and annotation

Labels are measurements from humans or systems, so their noise must be measured too.

Annotation quality is measured through vote fractions, gold accuracy, and agreement beyond chance. This notebook aggregates repeated labels and carries uncertainty into downstream training.

Save a copy to Drive to edit.

In [ ]:

import math
import random

import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(18)
random.seed(18)


def dataquality_ladder():
    """D1..D5 over real Breast Cancer with progressively worse data quality."""
    bc = load_breast_cancer()
    X0 = bc.data.astype(float)
    y0 = bc.target
    rng = np.random.default_rng(18)
    rungs = []

    rungs.append(("D1 clean", X0.copy(), y0.copy()))

    y2 = y0.copy()
    flip = rng.random(y2.shape) < 0.15
    y2[flip] = 1 - y2[flip]
    rungs.append(("D2 15% label noise", X0.copy(), y2))

    keep_pos = np.where(y0 == 1)[0]
    keep_neg = np.where(y0 == 0)[0][:30]
    idx = np.concatenate([keep_pos, keep_neg])
    rungs.append(("D3 class imbalance", X0[idx].copy(), y0[idx].copy()))

    X4 = X0 + rng.normal(0.0, X0.std(axis=0) * 0.5, size=X0.shape)
    rungs.append(("D4 heavy feature noise", X4, y0.copy()))

    X5 = X0.copy()
    holes = rng.random(X5.shape) < 0.2
    X5[holes] = np.nan
    col_means = np.nanmean(X5, axis=0)
    X5 = np.where(np.isnan(X5), col_means, X5)
    rungs.append(("D5 20% missing (mean-imputed)", X5, y0.copy()))

    return rungs


def split_scale(X, y, seed=0):
    x_train, x_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.35,
        random_state=seed,
        stratify=y,
    )
    scaler = StandardScaler()
    x_train = scaler.fit_transform(x_train)
    x_test = scaler.transform(x_test)
    return x_train, x_test, y_train, y_test


def fit_logistic(x_train, y_train, sample_weight=None):
    model = LogisticRegression(max_iter=2000, solver="lbfgs")
    model.fit(x_train, y_train, sample_weight=sample_weight)
    return model


def preview_ladder(rungs):
    for i, (name, X, y) in enumerate(rungs, start=1):
        counts = dict(zip(*np.unique(y, return_counts=True)))
        print(f"{i}. {name}: X={X.shape}, class_counts={counts}")
    print("sample rows", np.round(rungs[0][1][:3, :4], 3).tolist())


def plot_rungs_and_metric(rungs, rows, title, ylabel="accuracy"):
    fig, axes = plt.subplots(2, 5, figsize=(16, 6))
    metrics = [row["metric"] for row in rows]
    for ax, (name, X, y), metric in zip(axes[0], rungs, metrics):
        ax.scatter(X[:, 0], X[:, 1], c=y, s=12, cmap="coolwarm", alpha=0.65)
        ax.set_title(f"{name}\n{metric:.3f}")
        ax.set_xticks([])
        ax.set_yticks([])
    axes[1][0].plot(range(1, 6), metrics, marker="o")
    axes[1][0].set_title(title)
    axes[1][0].set_xlabel("data-quality rung")
    axes[1][0].set_ylabel(ylabel)
    axes[1][0].set_ylim(0.0, 1.05)
    for ax in axes[1][1:]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()


def simulate_annotators(y, seed=0):
    rng = np.random.default_rng(seed)
    annotators = []
    accuracies = [0.86, 0.82, 0.78]
    for acc in accuracies:
        labels = y.copy()
        flip = rng.random(y.shape) > acc
        labels[flip] = 1 - labels[flip]
        annotators.append(labels)
    return np.column_stack(annotators)


## The concept, built once (D1)

For item $i$ with $K$ annotators, the vote fraction is $v_i=rac{1}{K}\sum_k a_{ik}$ and the majority label is $\mathbf{1}[v_i\ge 0.5]$. Cohen's kappa is $\kappa=(p_o-p_e)/(1-p_e)$, where $p_e$ comes from annotator marginals.

In [ ]:
def cohen_kappa(a, b):
    a = np.asarray(a)
    b = np.asarray(b)
    observed = np.mean(a == b)
    pa = np.mean(a)
    pb = np.mean(b)
    expected = pa * pb + (1 - pa) * (1 - pb)
    return (observed - expected) / (1 - expected)


def aggregate_labels_train(X, y, seed=0, downweight_ambiguous=True):
    votes = simulate_annotators(y, seed=seed)
    vote_fraction = votes.mean(axis=1)
    labels = (vote_fraction >= 0.5).astype(int)
    confidence = np.abs(vote_fraction - 0.5) * 2
    sample_weight = 0.25 + confidence if downweight_ambiguous else np.ones_like(confidence)
    x_train, x_test, y_train, y_test = split_scale(X, y, seed=seed)
    label_train, label_test = train_test_split(labels, test_size=0.35, random_state=seed, stratify=y)
    weight_train, weight_test = train_test_split(sample_weight, test_size=0.35, random_state=seed, stratify=y)
    model = fit_logistic(x_train, label_train, sample_weight=weight_train)
    preds = model.predict(x_test)
    downstream = accuracy_score(y_test, preds)
    label_quality = accuracy_score(y_test, label_test, sample_weight=weight_test)
    return downstream * label_quality

Assert the lesson's vote fractions, majority labels, gold accuracy, and kappa.

In [ ]:
votes = np.array([
    [1, 1, 0],
    [1, 0, 0],
    [0, 0, 0],
    [1, 1, 1],
    [0, 1, 0],
])
fractions = votes.mean(axis=1)
majority = (fractions >= 0.5).astype(int)
gold = np.array([1, 0, 0, 1, 0])
rater_acc = [(votes[:, j] == gold).mean() for j in range(3)]
a = np.array([1, 1, 0, 1, 0])
b = np.array([1, 0, 0, 1, 1])
kappa = cohen_kappa(a, b)
print(np.round(fractions, 3).tolist(), majority.tolist())
print(np.round(rater_acc, 3).tolist(), round(kappa, 3))
assert np.round(fractions, 3).tolist() == [0.667, 0.333, 0.0, 1.0, 0.333]
assert majority.tolist() == [1, 0, 0, 1, 0]
assert np.round(rater_acc, 3).tolist() == [0.8, 0.8, 0.8]
assert round(kappa, 3) == 0.167

## The dataset ladder

Every notebook uses the same real Breast Cancer base table and changes data quality only: clean, label-noisy, imbalanced, feature-noisy, then missing-and-imputed. The model stays fixed so movement in the curve is caused by the data.

In [ ]:
rungs = dataquality_ladder()
preview_ladder(rungs)

In [ ]:
rows = []
for name, X, y in rungs:
    metric = aggregate_labels_train(X, y, seed=5, downweight_ambiguous=True)
    rows.append({"name": name, "metric": metric})
    print(f"{name:28s} label_adjusted_accuracy={metric:.3f}")

In [ ]:
plot_rungs_and_metric(rungs, rows, "Label-quality-adjusted accuracy")

## Pitfall on D5: counting majority vote as ground truth

Majority vote discards uncertainty. The fix is to review or down-weight low-confidence items using vote fraction distance from 0.5.

In [ ]:
name, X, y = rungs[-1]
plain = aggregate_labels_train(X, y, seed=7, downweight_ambiguous=False)
weighted = aggregate_labels_train(X, y, seed=7, downweight_ambiguous=True)
votes = simulate_annotators(y, seed=7)
fractions = votes.mean(axis=1)
ambiguous = np.mean((fractions > 0.25) & (fractions < 0.75))
print(f"ambiguous fraction={ambiguous:.3f}")
print(f"plain majority metric={plain:.3f} downweighted metric={weighted:.3f}")
assert ambiguous > 0.0
assert weighted >= plain - 0.08

## Evaluate it + Practice

- Metric: downstream accuracy multiplied by held-out label quality; compare with the no-skill majority-class baseline.
- Cheap sanity check: three lesson raters should each score 0.800 on gold.
- Ablation: train with all vote fractions as weight 1 and compare ambiguity robustness.
- Failure signals: kappa near zero, many vote fractions near 0.5, or low gold-rater accuracy.

Practice prompts:
1. Change one corruption level in `dataquality_ladder()` and predict the metric direction.


In [ ]:
# Your turn: change one corruption and rerun the ladder table.


2. Add one slice check that could catch a global-average pitfall.

In [ ]:
# Your turn: define a slice and compare its metric with the global metric.


3. Replace the fixed logistic model with another CPU-safe classifier and explain whether the data-quality ordering changed.

In [ ]:
# Your turn: try a second model without changing the data-cleaning method.
